# Jazan City — Points of Interest Mapping
**Author:** Ali Abutaleb, PhD | George Mason University  
**Data Sources:** Google Maps Places API, OpenStreetMap  

This notebook collects, processes, and visualizes Points of Interest (POI) in Jazan City, Saudi Arabia.  
Categories covered: **Restaurants**, **Cafes**, and **Schools**.

---

## Step 1 — Import Libraries
We import all required Python libraries for mapping, spatial analysis, and data collection.

In [ ]:
import time
import numpy as np
import osmnx as ox
import geopandas as gpd
import folium
import googlemaps
import subprocess
from shapely.ops import unary_union
from folium import FeatureGroup, LayerControl
from folium.plugins import HeatMap, MiniMap, Fullscreen

print("All libraries imported successfully!")

## Step 2 — Configure Google Maps API
We connect to the Google Maps Places API using an API key.  
This allows us to search for businesses and places across Jazan City.

In [ ]:
# Replace with your own Google Maps API key
API_KEY = "YOUR_API_KEY_HERE"
gmaps = googlemaps.Client(key=API_KEY)

print("Google Maps API connected!")

## Step 3 — Define the Search Grid
Since Google Maps API only returns 60 results per search, we create a grid of search points  
covering all of Jazan City. Each point searches within a 1.5km radius.

**Jazan City Center Coordinates:** 16.8892°N, 42.5511°E

In [ ]:
# Create a grid of latitude/longitude points covering Jazan City
# Points are spaced 0.02 degrees apart (roughly 2km)
lat_points = np.arange(16.85, 16.95, 0.02)
lng_points = np.arange(42.50, 42.62, 0.02)

print(f"Grid created: {len(lat_points)} x {len(lng_points)} = {len(lat_points) * len(lng_points)} search points")

## Step 4 — Define Data Collection Functions
We define two helper functions:
- `get_all_results()`: Fetches all pages of results for a given place type or keyword (up to 60 results per point)
- Includes automatic retry logic in case of connection errors

In [ ]:
def get_all_results(gmaps, location, radius, place_type=None, keyword=None, retries=3):
    """
    Fetches all results from Google Places API for a given location.
    Handles pagination (up to 3 pages = 60 results) and retries on failure.
    
    Args:
        gmaps: Google Maps client
        location: Tuple of (latitude, longitude)
        radius: Search radius in meters
        place_type: Google place type (e.g. 'restaurant', 'cafe')
        keyword: Search keyword in English or Arabic
        retries: Number of retry attempts on failure
    
    Returns:
        List of place results
    """
    for attempt in range(retries):
        try:
            # Search by type or keyword
            if place_type:
                results = gmaps.places_nearby(location=location, radius=radius, type=place_type)
            else:
                results = gmaps.places_nearby(location=location, radius=radius, keyword=keyword)

            all_results = results["results"]

            # Paginate through additional results (Google returns max 3 pages)
            while "next_page_token" in results:
                time.sleep(2)  # Required delay between page requests
                results = gmaps.places_nearby(page_token=results["next_page_token"])
                all_results.extend(results["results"])

            return all_results

        except Exception as e:
            print(f"  Error: {e} — Retrying in 5 seconds... (attempt {attempt + 1}/{retries})")
            time.sleep(5)

    return []  # Return empty list if all retries fail


print("Helper functions defined!")

## Step 5 — Collect Points of Interest
We search for restaurants, cafes, and schools across the Jazan City grid.  
Each category is searched using both **English** and **Arabic** keywords to maximize coverage.

> **Note:** This step makes many API calls and may take several minutes to complete.

In [ ]:
# Initialize storage containers
all_places = {"restaurants": [], "cafes": [], "schools": []}
seen_ids = set()  # Track unique place IDs to avoid duplicates

# Define search terms for each category
# Each entry: (place_type, arabic_keyword, category)
searches = [
    ("restaurant", "مطعم",  "restaurants"),   # Restaurants in English + Arabic
    ("cafe",       "كافيه", "cafes"),          # Cafes in English + Arabic
    ("school",     "مدرسة", "schools"),        # Schools in English + Arabic
]

total = len(lat_points) * len(lng_points) * len(searches) * 2  # x2 for type + keyword
count = 0

# Loop through all grid points and search categories
for lat in lat_points:
    for lng in lng_points:
        for place_type, keyword, category in searches:

            # Search by place type (English)
            places = get_all_results(gmaps, (lat, lng), 1500, place_type=place_type)
            for place in places:
                if place["place_id"] not in seen_ids:
                    seen_ids.add(place["place_id"])
                    all_places[category].append(place)
            count += 1
            print(f"Progress: {count}/{total}", end="\r")
            time.sleep(1)

            # Search by Arabic keyword
            places = get_all_results(gmaps, (lat, lng), 1500, keyword=keyword)
            for place in places:
                if place["place_id"] not in seen_ids:
                    seen_ids.add(place["place_id"])
                    all_places[category].append(place)
            count += 1
            print(f"Progress: {count}/{total}", end="\r")
            time.sleep(1)

print("\n\nData collection complete!")
print(f"Restaurants found : {len(all_places['restaurants'])}")
print(f"Cafes found       : {len(all_places['cafes'])}")
print(f"Schools found     : {len(all_places['schools'])}")

## Step 6 — Supplement with Street-Level Searches
Some places in Jazan's main streets are not captured by the grid search.  
We add targeted text searches for key streets to fill the gaps.

In [ ]:
# Street-level text searches to supplement the grid
# Format: (search query, category)
street_searches = [
    # Al-Madinah Al-Munawwarah Street
    ("coffee shop المدينة المنورة جازان",          "cafes"),
    ("كافيه شارع المدينة المنورة جازان",           "cafes"),
    ("مقهى شارع المدينة المنورة جازان",            "cafes"),
    ("قهوة شارع المدينة المنورة جازان",            "cafes"),
    ("restaurant شارع المدينة المنورة جازان",       "restaurants"),
    ("مطعم شارع المدينة المنورة جازان",            "restaurants"),

    # Jazan Corniche
    ("كافيه كورنيش جازان",                        "cafes"),
    ("مقهى كورنيش جازان",                         "cafes"),
    ("مطعم كورنيش جازان",                         "restaurants"),
    ("restaurant كورنيش جازان",                    "restaurants"),

    # King Fahd Street
    ("كافيه شارع الملك فهد جازان",                "cafes"),
    ("مطعم شارع الملك فهد جازان",                 "restaurants"),

    # Prince Mohammed Street
    ("كافيه شارع الأمير محمد جازان",              "cafes"),
    ("مطعم شارع الأمير محمد جازان",               "restaurants"),

    # City Center
    ("كافيه وسط جازان",                           "cafes"),
    ("مطعم وسط جازان",                            "restaurants"),
]

added = {"restaurants": 0, "cafes": 0, "schools": 0}

for query, category in street_searches:
    try:
        results = gmaps.places(query=query)
        for place in results["results"]:
            if place["place_id"] not in seen_ids:
                seen_ids.add(place["place_id"])
                all_places[category].append(place)
                added[category] += 1
        time.sleep(1)
    except Exception as e:
        print(f"Error on '{query}': {e}")
        time.sleep(3)

print("Street-level search complete!")
print(f"New restaurants added : {added['restaurants']}")
print(f"New cafes added       : {added['cafes']}")
print(f"\nUpdated totals:")
print(f"Restaurants : {len(all_places['restaurants'])}")
print(f"Cafes       : {len(all_places['cafes'])}")
print(f"Schools     : {len(all_places['schools'])}")

## Step 7 — Process and Filter Data
We extract coordinates and metadata from the raw Google API results,  
filtering out any places that fall outside the Jazan City boundaries.

In [ ]:
# Jazan City bounding box (keeps only places within the city)
JAZAN_LAT_MIN = 16.80
JAZAN_LAT_MAX = 17.00
JAZAN_LNG_MIN = 42.45
JAZAN_LNG_MAX = 42.65


def to_points_google(places):
    """
    Converts raw Google API results into a clean list of coordinate dictionaries.
    Filters out places outside the Jazan City bounding box.
    Also extracts rating, review count, and address for map popups.
    """
    coords = []
    for place in places:
        try:
            lat = place["geometry"]["location"]["lat"]
            lng = place["geometry"]["location"]["lng"]

            # Only keep places inside Jazan City
            if JAZAN_LAT_MIN < lat < JAZAN_LAT_MAX and JAZAN_LNG_MIN < lng < JAZAN_LNG_MAX:
                coords.append({
                    "name":          place["name"],
                    "lat":           lat,
                    "lng":           lng,
                    "rating":        place.get("rating", 0),
                    "total_ratings": place.get("user_ratings_total", 0),
                    "address":       place.get("vicinity", "N/A")
                })
        except:
            pass
    return coords


# Process each category
restaurants_pts = to_points_google(all_places["restaurants"])
cafes_pts       = to_points_google(all_places["cafes"])
schools_pts     = to_points_google(all_places["schools"])

# Calculate map center and bounding box from all collected points
all_points = restaurants_pts + cafes_pts + schools_pts
all_lats   = [p["lat"] for p in all_points]
all_lngs   = [p["lng"] for p in all_points]

center     = [sum(all_lats) / len(all_lats), sum(all_lngs) / len(all_lngs)]
fit_bounds = [[min(all_lats), min(all_lngs)], [max(all_lats), max(all_lngs)]]

print("Data processed and filtered!")
print(f"Restaurants in Jazan : {len(restaurants_pts)}")
print(f"Cafes in Jazan       : {len(cafes_pts)}")
print(f"Schools in Jazan     : {len(schools_pts)}")
print(f"Map center           : {center}")

## Step 8 — Load City Boundary
We fetch the administrative boundary of Jazan City from OpenStreetMap (OSM).  
This will be displayed as a white outline on the final map.

In [ ]:
print("Loading Jazan city boundary from OpenStreetMap...")

try:
    # Query OSM for administrative boundaries near Jazan city center
    jazan_boundary = ox.features_from_point(
        (16.8892, 42.5511),
        tags={"boundary": "administrative", "admin_level": "8"},
        dist=5000
    )

    # Merge all sub-district polygons into one clean outer boundary
    merged = unary_union(jazan_boundary.geometry)
    jazan_boundary_outer = gpd.GeoDataFrame(
        geometry=[merged.convex_hull],
        crs=jazan_boundary.crs
    )

    has_boundary = True
    print("City boundary loaded successfully!")

except Exception as e:
    print(f"Could not load boundary: {e}")
    print("A circle will be used as a fallback boundary.")
    has_boundary = False

## Step 9 — Build the Interactive Map
We assemble the final interactive map with:
- **Dark basemap** (CartoDB Dark Matter)
- **Color-coded layers** for each POI category
- **Size by rating** (larger dot = higher-rated place)
- **Clickable popups** with name, address, and rating
- **Heatmap** showing restaurant density
- **Layer controls** to toggle categories on/off
- **Minimap**, **north arrow**, **title**, **legend**, and **credits**

In [ ]:
# ─── Helper Functions ───────────────────────────────────────────────────────

def get_radius(rating):
    """Returns circle radius based on place rating (higher rating = larger dot)."""
    if rating >= 4.5:  return 8
    elif rating >= 4.0: return 6
    elif rating >= 3.0: return 5
    else:               return 4


def make_popup(place, emoji):
    """Creates an HTML popup showing place name, address, and rating."""
    rating  = place.get("rating", "N/A")
    total   = place.get("total_ratings", 0)
    address = place.get("address", "N/A")
    stars   = "⭐" * int(rating) if isinstance(rating, (int, float)) else ""
    return folium.Popup(
        f'<div style="font-family:Arial;min-width:150px;">'
        f'<b style="font-size:13px;">{emoji} {place["name"]}</b><br>'
        f'<span style="color:#555;font-size:11px;">{address}</span><br>'
        f'<span style="font-size:12px;">{stars} {rating} ({total} reviews)</span>'
        f'</div>',
        max_width=250
    )


# ─── Create Base Map ─────────────────────────────────────────────────────────

m = folium.Map(
    location=center,
    zoom_start=13,
    tiles="CartoDB dark_matter",
    control_scale=True
)
m.fit_bounds(fit_bounds)

# Add fullscreen button
Fullscreen(position="topright").add_to(m)

# Add minimap (overview in bottom right)
MiniMap(
    tiles="CartoDB dark_matter",
    position="bottomright",
    width=150, height=150,
    zoom_level_offset=-6
).add_to(m)


# ─── Map Title (top left) ─────────────────────────────────────────────────────

title_html = (
    '<div style="position:fixed;top:20px;left:60px;z-index:1000;'
    'background-color:rgba(0,0,0,0.75);padding:14px 20px;border-radius:12px;'
    'border:1px solid #ffffff33;">'
    '<h2 style="color:white;margin:0;font-size:18px;font-family:Arial;letter-spacing:1px;">🗺️ JAZAN CITY</h2>'
    '<p style="color:#cccccc;margin:4px 0 0 0;font-size:12px;font-family:Arial;letter-spacing:2px;">POINTS OF INTEREST</p>'
    '<p style="color:#888888;margin:6px 0 0 0;font-size:11px;font-family:Arial;">Restaurants • Cafes • Schools</p>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(title_html))


# ─── Legend (bottom left) ─────────────────────────────────────────────────────

legend_html = (
    '<div style="position:fixed;bottom:30px;left:20px;z-index:1000;'
    'background-color:rgba(0,0,0,0.75);padding:16px 20px;border-radius:12px;'
    'border:1px solid #ffffff33;font-family:Arial;min-width:180px;">'
    '<p style="color:white;font-size:13px;font-weight:bold;margin:0 0 12px 0;letter-spacing:1px;">LEGEND</p>'
    f'<p style="color:white;font-size:12px;margin:6px 0;">'
    f'<span style="color:#FF6B6B;font-size:18px;">●</span> Restaurants '
    f'<span style="color:#888;font-size:11px;">({len(restaurants_pts)})</span></p>'
    f'<p style="color:white;font-size:12px;margin:6px 0;">'
    f'<span style="color:#FFD93D;font-size:18px;">●</span> Cafes '
    f'<span style="color:#888;font-size:11px;">({len(cafes_pts)})</span></p>'
    f'<p style="color:white;font-size:12px;margin:6px 0;">'
    f'<span style="color:#6BCB77;font-size:18px;">●</span> Schools '
    f'<span style="color:#888;font-size:11px;">({len(schools_pts)})</span></p>'
    '<hr style="border:0;border-top:1px solid #ffffff22;margin:10px 0;">'
    '<p style="color:#aaaaaa;font-size:11px;margin:4px 0;">⭐ Larger dot = Higher rating</p>'
    '<p style="color:#aaaaaa;font-size:11px;margin:4px 0;">🖱️ Click dot for details</p>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))


# ─── Credits (bottom right, above minimap) ───────────────────────────────────

credits_html = (
    '<div style="position:fixed;bottom:220px;right:20px;z-index:1000;'
    'background-color:rgba(0,0,0,0.75);padding:12px 16px;border-radius:12px;'
    'border:1px solid #ffffff33;font-family:Arial;text-align:right;">'
    '<p style="color:white;font-size:13px;margin:0;font-weight:bold;letter-spacing:0.5px;">Ali Abutaleb, PhD</p>'
    '<div style="border-top:1px solid #ffffff22;margin:6px 0;"></div>'
    '<p style="color:#aaaaaa;font-size:11px;margin:0;">George Mason University</p>'
    '<p style="color:#555555;font-size:10px;margin:4px 0 0 0;">Data: Google Maps API • OSM</p>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(credits_html))


# ─── North Arrow ─────────────────────────────────────────────────────────────

north_arrow_html = (
    '<div style="position:fixed;top:120px;left:20px;z-index:1000;'
    'background-color:rgba(0,0,0,0.75);padding:8px;border-radius:8px;'
    'border:1px solid #ffffff33;text-align:center;font-family:Arial;">'
    '<div style="color:white;font-size:20px;line-height:1;">⬆</div>'
    '<div style="color:#aaaaaa;font-size:10px;">N</div>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(north_arrow_html))


# ─── City Boundary Layer ──────────────────────────────────────────────────────

boundary_layer = FeatureGroup(name="🏙️ City Boundary", show=True)

if has_boundary:
    folium.GeoJson(
        jazan_boundary_outer,
        style_function=lambda feature: {
            "fillColor": "#ffffff",
            "color":     "#ffffff",
            "weight":    2,
            "fillOpacity": 0.03,
            "dashArray": "5, 5"
        }
    ).add_to(boundary_layer)
else:
    # Fallback: draw a 5km circle around the city center
    folium.Circle(
        location=[16.8892, 42.5511],
        radius=5000, color="white",
        weight=2, fill=True, fill_opacity=0.03
    ).add_to(boundary_layer)

boundary_layer.add_to(m)


# ─── Restaurants Layer (red dots) ────────────────────────────────────────────

restaurants_layer = FeatureGroup(name="🔴 Restaurants", show=True)
for place in restaurants_pts:
    folium.CircleMarker(
        location=[place["lat"], place["lng"]],
        radius=get_radius(place["rating"]),
        color="#FF6B6B", fill=True, fill_color="#FF6B6B",
        fill_opacity=0.8, weight=1,
        popup=make_popup(place, "🍽️")
    ).add_to(restaurants_layer)
restaurants_layer.add_to(m)


# ─── Cafes Layer (yellow dots) ───────────────────────────────────────────────

cafes_layer = FeatureGroup(name="🟡 Cafes", show=True)
for place in cafes_pts:
    folium.CircleMarker(
        location=[place["lat"], place["lng"]],
        radius=get_radius(place["rating"]),
        color="#FFD93D", fill=True, fill_color="#FFD93D",
        fill_opacity=0.8, weight=1,
        popup=make_popup(place, "☕")
    ).add_to(cafes_layer)
cafes_layer.add_to(m)


# ─── Schools Layer (green dots) ──────────────────────────────────────────────

schools_layer = FeatureGroup(name="🟢 Schools", show=True)
for place in schools_pts:
    folium.CircleMarker(
        location=[place["lat"], place["lng"]],
        radius=get_radius(place["rating"]),
        color="#6BCB77", fill=True, fill_color="#6BCB77",
        fill_opacity=0.8, weight=1,
        popup=make_popup(place, "🏫")
    ).add_to(schools_layer)
schools_layer.add_to(m)


# ─── Heatmap Layer (restaurant density, hidden by default) ───────────────────

heat_data = [[p["lat"], p["lng"]] for p in restaurants_pts]
heatmap_layer = FeatureGroup(name="🔥 Restaurant Heatmap", show=False)
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.4).add_to(heatmap_layer)
heatmap_layer.add_to(m)


# ─── Layer Control Panel (top right) ─────────────────────────────────────────

LayerControl(collapsed=False).add_to(m)


# ─── Save and Open ────────────────────────────────────────────────────────────

m.save("jazan_map.html")
subprocess.run(["open", "jazan_map.html"])

print("Map saved and opened!")
print(f"Restaurants : {len(restaurants_pts)}")
print(f"Cafes       : {len(cafes_pts)}")
print(f"Schools     : {len(schools_pts)}")

---
## Map is ready!
The interactive map `jazan_map.html` has been saved in your project folder.  
Open it in any browser to explore Jazan City's Points of Interest.

**Ali Abutaleb, PhD** | George Mason University